# A1 · A Pipeline from scratch: who calls `fit` and who calls `transform`

**Type**: Implement from scratch · **Level**: ●●○ independent · **Time**: about 35 minutes · **core path ★**

## Objective

Implement a `MyPipeline` class with `fit`, `predict`, `predict_proba`, `named_steps` and `set_params`, check it against
`sklearn.pipeline.Pipeline` and show, by counting the calls, that in `predict` no intermediate step is refitted.

## Dataset

`ds.titanic()`: the five numeric columns (`pclass`, `age`, `siblings_spouses`, `parents_children`, `fare`; `age` has missing values)
and the target `survived`. Stratified 75/25 split. The `Spy` class (defined in the setup) wraps a scaler and counts
how many times `fit_transform` and `transform` are called.

## Educational purpose

Take the `Pipeline` apart to understand that it has nothing magic: it is a loop that calls `fit_transform` on the intermediate steps in
training and `transform` in prediction. All the protection against leakage comes from this simple rule.

## Task

1. Write `MyPipeline(steps)`, where `steps` is a list of pairs `(name, object)`. In `__init__` save `steps` and create `named_steps`, a dictionary name → object.
2. `fit(X, y)`: for every step **except the last** replace `X` with `step.fit_transform(X, y)`; then call `fit(X, y)` on the last one. Return `self`.
3. `predict(X)` and `predict_proba(X)`: for every step except the last apply **only** `transform`; then call the method of the last one.
4. `set_params(**kw)`: for every key `"step__parameter"` call `set_params(parameter=value)` on the right step. Return `self`.
5. Build `mine` with three steps (`imputer`, `scaler` = `Spy(StandardScaler())`, `model`), train it on `X_tr`, call `predict_proba` on `X_te` **three times** and save in `spy_fit_transform` and `spy_transform` the counters of the scaler.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

df = ds.titanic()
NUM = ["pclass", "age", "siblings_spouses", "parents_children", "fare"]
X_tr, X_te, y_tr, y_te = train_test_split(df[NUM], df["survived"], test_size=0.25, stratify=df["survived"], random_state=0)


class Spy:
    """Wraps a transformer and counts the calls to fit_transform and transform."""

    def __init__(self, t):
        self.t, self.n_fit_transform, self.n_transform = t, 0, 0

    def fit_transform(self, X, y=None):
        self.n_fit_transform += 1
        return self.t.fit_transform(X, y)

    def transform(self, X):
        self.n_transform += 1
        return self.t.transform(X)

    def fit(self, X, y=None):                      # a correct pipeline must never use it
        raise RuntimeError("the pipeline must not call fit on the intermediate steps")

## Your work

In [ ]:
class MyPipeline:
    def __init__(self, steps):
        self.steps = steps
        self.named_steps = ...

    def fit(self, X, y):
        ...
        return self

    def predict(self, X):
        ...

    def predict_proba(self, X):
        ...

    def set_params(self, **kw):
        ...
        return self

In [ ]:
spy = Spy(StandardScaler())
mine = MyPipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", spy),
                   ("model", LogisticRegression(max_iter=1000))])
mine.fit(X_tr, y_tr)
for _ in range(3):
    p = mine.predict_proba(X_te)
spy_fit_transform = ...
spy_transform = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`fit`: `Xt = X; for name, step in self.steps[:-1]: Xt = step.fit_transform(Xt, y)`; then `self.steps[-1][1].fit(Xt, y)`. For `set_params`: `name, parameter = key.split("__", 1)` and `self.named_steps[name].set_params(**{parameter: value})`.'):
    ref = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
                    ("model", LogisticRegression(max_iter=1000))]).fit(X_tr, y_tr)
    check(close(mine.predict_proba(X_te), ref.predict_proba(X_te)), "the probabilities match those of sklearn.Pipeline",
          "MyPipeline.predict_proba must give the same result as Pipeline")
    check(close(mine.predict(X_te), ref.predict(X_te)), "the predicted classes match", "predict must use the same transformations")
    check(list(mine.named_steps) == ["imputer", "scaler", "model"], "named_steps is a dictionary name -> object", "named_steps = dict(steps)")
    check(spy_fit_transform == 1, "in training the scaler did fit_transform just once", "fit_transform must be called once, in fit")
    check(spy_transform == 3, "in the three predictions the scaler did only transform (3 times)", "in predict only transform is called: once per prediction")
    mine.set_params(model__C=0.01)
    check(mine.named_steps["model"].C == 0.01, "set_params reaches the parameters of the last step", "set_params('model__C') must modify the model")
    mine.fit(X_tr, y_tr)
    check(not np.allclose(mine.predict_proba(X_te), ref.predict_proba(X_te)), "after set_params and a new fit the model changes", "small C = strong regularization: the probabilities change")
done()

---
The commented solution is in `solutions/A_from_scratch/A1_pipeline_from_scratch.ipynb`. Open it after you pass the check, or after you have really tried.